# 导入必要的库
导入数据处理和可视化所需的库，包括rasterio用于读取.tif文件，matplotlib用于绘图，numpy用于数组处理，以及geopandas用于地理数据处理。

In [4]:
# 导入必要的库
import rasterio  # 用于读取.tif文件
import matplotlib.pyplot as plt  # 用于绘图
import numpy as np  # 用于数组处理
import geopandas as gpd  # 用于地理数据处理

ModuleNotFoundError: No module named 'rasterio'

# 读取地理数据文件
使用rasterio读取.tif文件，获取栅格数据和元数据信息。处理文件路径并加载数据。

In [ ]:
# 读取地理数据文件
file_path = r'F:\ScientificDatabase\electricity_downscaling_data\China_1km_Ele_201204_201912\China_1km_Ele_201204_201912\201801.tif'

# 使用rasterio打开.tif文件
with rasterio.open(file_path) as dataset:
    # 获取栅格数据
    raster_data = dataset.read(1)
    
    # 获取元数据信息
    metadata = dataset.meta

# 打印元数据信息
print("Metadata:", metadata)

# 显示栅格数据的基本信息
print("Raster data shape:", raster_data.shape)
print("Raster data type:", raster_data.dtype)

# 数据基本信息查看
检查数据的基本属性，包括空间范围、投影信息、分辨率等。展示数据的统计特征。

In [ ]:
# 检查数据的空间范围、投影信息、分辨率等
with rasterio.open(file_path) as dataset:
    bounds = dataset.bounds
    crs = dataset.crs
    transform = dataset.transform
    resolution = dataset.res

# 打印空间范围、投影信息、分辨率等
print("Bounds:", bounds)
print("CRS:", crs)
print("Transform:", transform)
print("Resolution:", resolution)

# 计算并展示数据的统计特征
raster_min = np.min(raster_data)
raster_max = np.max(raster_data)
raster_mean = np.mean(raster_data)
raster_std = np.std(raster_data)

print("Min value:", raster_min)
print("Max value:", raster_max)
print("Mean value:", raster_mean)
print("Standard deviation:", raster_std)

# 数据预处理与转换
对数据进行必要的预处理，包括缺失值处理、投影变换、数据归一化等操作。

In [ ]:
# 数据预处理与转换

# 处理缺失值，将缺失值替换为0
raster_data = np.nan_to_num(raster_data, nan=0.0)

# 数据归一化，将数据缩放到0-1范围
raster_data_normalized = (raster_data - raster_min) / (raster_max - raster_min)

# 投影变换（如果需要）
# 这里假设目标投影为WGS84
target_crs = 'EPSG:4326'
if crs != target_crs:
    with rasterio.open(file_path) as src:
        transform, width, height = rasterio.warp.calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds)
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': target_crs,
            'transform': transform,
            'width': width,
            'height': height
        })

        with rasterio.open('transformed.tif', 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                rasterio.warp.reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=target_crs,
                    resampling=rasterio.warp.Resampling.nearest)
    
    # 读取转换后的数据
    with rasterio.open('transformed.tif') as dataset:
        raster_data_transformed = dataset.read(1)
else:
    raster_data_transformed = raster_data_normalized

# 打印转换后的数据统计特征
raster_min_transformed = np.min(raster_data_transformed)
raster_max_transformed = np.max(raster_data_transformed)
raster_mean_transformed = np.mean(raster_data_transformed)
raster_std_transformed = np.std(raster_data_transformed)

print("Transformed Min value:", raster_min_transformed)
print("Transformed Max value:", raster_max_transformed)
print("Transformed Mean value:", raster_mean_transformed)
print("Transformed Standard deviation:", raster_std_transformed)

# 基础地图可视化
使用matplotlib创建基础地图，设置适当的颜色映射方案，展示栅格数据。

In [ ]:
# 基础地图可视化

# 创建一个图形对象和子图
fig, ax = plt.subplots(figsize=(10, 10))

# 使用imshow函数展示栅格数据
cax = ax.imshow(raster_data_transformed, cmap='viridis')

# 添加颜色条
cbar = fig.colorbar(cax, ax=ax, orientation='vertical', shrink=0.5)
cbar.set_label('Normalized Value')

# 设置标题和坐标轴标签
ax.set_title('基础地图可视化')
ax.set_xlabel('X坐标')
ax.set_ylabel('Y坐标')

# 显示图形
plt.show()

# 添加地图要素
添加地图必要元素，如色标、比例尺、指北针等。优化地图布局和样式。

In [ ]:
# 添加地图要素

# 创建一个图形对象和子图
fig, ax = plt.subplots(figsize=(10, 10))

# 使用imshow函数展示栅格数据
cax = ax.imshow(raster_data_transformed, cmap='viridis')

# 添加颜色条
cbar = fig.colorbar(cax, ax=ax, orientation='vertical', shrink=0.5)
cbar.set_label('Normalized Value')

# 添加比例尺
scalebar = AnchoredSizeBar(ax.transData,
                           100, '100 m', 'lower right', 
                           pad=0.1,
                           color='black',
                           frameon=False,
                           size_vertical=1)
ax.add_artist(scalebar)

# 添加指北针
x, y, arrow_length = 0.1, 0.1, 0.05
ax.annotate('N', xy=(x, y), xytext=(x, y - arrow_length),
            arrowprops=dict(facecolor='black', width=5, headwidth=15),
            ha='center', va='center', fontsize=12,
            xycoords=ax.transAxes)

# 设置标题和坐标轴标签
ax.set_title('添加地图要素')
ax.set_xlabel('X坐标')
ax.set_ylabel('Y坐标')

# 显示图形
plt.show()

# 数据统计分析
计算并可视化数据的统计特征，如直方图、空间分布特征等。

In [ ]:
# 数据统计分析

# 计算数据的直方图
hist, bin_edges = np.histogram(raster_data_transformed, bins=50)

# 创建一个图形对象和子图
fig, ax = plt.subplots(figsize=(10, 6))

# 绘制直方图
ax.bar(bin_edges[:-1], hist, width=np.diff(bin_edges), edgecolor='black')

# 设置标题和坐标轴标签
ax.set_title('数据直方图')
ax.set_xlabel('值')
ax.set_ylabel('频数')

# 显示图形
plt.show()

# 可视化数据的空间分布特征
fig, ax = plt.subplots(figsize=(10, 10))

# 使用imshow函数展示栅格数据
cax = ax.imshow(raster_data_transformed, cmap='viridis')

# 添加颜色条
cbar = fig.colorbar(cax, ax=ax, orientation='vertical', shrink=0.5)
cbar.set_label('Normalized Value')

# 设置标题和坐标轴标签
ax.set_title('数据空间分布特征')
ax.set_xlabel('X坐标')
ax.set_ylabel('Y坐标')

# 显示图形
plt.show()